In [18]:
import os
import time
from clearml import Task
from clearml.config import running_remotely
from loguru import logger
import json
import pymysql
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler, RobustScaler
import numpy as np
import shap
# [해결 포인트 1] PyTorch 표준 텐서보드 Writer 가져오기 (가장 확실한 대안)
# from torch.utils.tensorboard import SummaryWriter

# 환경 변수 선언 (Configuration)
# work_environ = "company"
work_environ = "home"
phase = os.environ.get("PHASE", "dev")
epochs_cnt = os.environ.get("EPOCHS_CNT", 4)
docker_image = os.environ.get("DOCKER_IMAGE", "172.16.11.236:5000/spire/python:3.12-bullseye")
model_type_str = os.environ.get("MODEL_TYPE_STR", "LSTMAe")
process_id = os.environ.get("PROCESS_ID", "P102")
sub_process_id = os.environ.get("SUB_PROCESS_ID", "1001")
project_name = os.environ.get("PROJECT_NAME", "mes")
if phase == "prod":
    db_conf = json.loads(os.environ.get("DB_CONF", '{"host":"172.16.9.60", "port": 3306, "user": "mlops_detect", "password": "QoS908Z1!", "database": "mes_metric"}'))
else:
    if work_environ == "company":
        db_passwd = "QoS908Z1!"
        db_host = "172.16.9.60"
    else:
        db_passwd = "QoS908Z1!"
        db_host = "172.21.144.1"
    db_con_str = '{'+f'"host":"{db_host}", "port": 3306, "user": "mlops_detect", "password": "{db_passwd}", "database": "mes_metric"'+'}'
    db_conf = json.loads(os.environ.get("DB_CONF", db_con_str))
task_name = f"{process_id}_{sub_process_id}"
BUCKET_NAME = "clearml-data"
data_begin_days = os.environ.get("DATA_BEGIN_DAYS", -1)
use_clearml = True if os.environ.get("USE_CLEARML", 0) == 1 else False
task_version = os.environ.get("TASK_VERSION", "1.0")

def db_get_columns(db_conf: dict, table_name: str):
    query = f"""describe {table_name}"""
    print(f'db_conf = {db_conf}')
    print(f'table_name = {table_name}')
    columns = []
    try:
        with pymysql.connect(host=db_conf['host'], port=db_conf['port'],
                             user=db_conf['user'], password=db_conf['password'],
                             database=db_conf['database']) as conn:
            print(f'row = {conn}')
            with conn.cursor() as cursor:
                print(f'row = {cursor}')
                # DESCRIBE 명령어로 테이블 정보 요청
                cursor.execute(query)
                rows = cursor.fetchall()
                for row in rows:
                    print(f'row = {row}')
                    if row[0] not in ("id", "process_id", "sub_process_id", 'is_anomaly_label', 'created_at'):
                        columns.append(row[0])
    except Exception as ex:
        logger.error(f"Exception: {ex}")
        columns = []
    return columns
logger.info(f'breaking_point #1')
feature_cols = db_get_columns(db_conf=db_conf, table_name=task_name)

n_features = len(feature_cols)
logger.info(f'breaking_point #2')
# ---------------------------------------------------------
# 1. 하이퍼파라미터 설정 (건조기 센서 데이터 최적 Baseline)
# ---------------------------------------------------------
lstmae_def_params = {
    'seq_len': 20,            # 시퀀스 길이 (예: 1분 단위 수집 시 1시간 분량)
    'n_features': n_features, # 센서 개수 (예: 온도, 습도, 진동)
    'inner_dim': 64,          # LSTM 은닉층 차원
    'bottleneck_dim': 16,     # 압축 차원 (특징 추출 공간)
    'batch_size': 64,         # 배치 크기
    'learning_rate': 0.001,   # 초기 학습률
    'dropout': 0.1,           # 시계열 정보 유지를 위한 낮은 드롭아웃
    'epochs': 20              # 학습 에폭 수
}
lstmae_params = json.loads(os.environ.get("MODEL_PARAMS", json.dumps(lstmae_def_params)))
for key in lstmae_def_params.keys():
    if key not in lstmae_params:
        lstmae_params[key] = lstmae_def_params[key]

data_def_params = {
    'split_ratio': 0.9
}
data_params = json.loads(os.environ.get("DATA_PARAMS", json.dumps(data_def_params)))
for key in data_def_params.keys():
    if key not in data_params:
        data_params[key] = data_def_params[key]
logger.info(f'breaking_point #3')
if phase != "prod":
    lstmae_params['epochs'] = epochs_cnt

if work_environ == "home":
    OBJECT_STORAGE_ENDPOINT = 'http://192.168.0.83:9000'  # Garage 서버 주소 (기본 포트 3900)
    AWS_ACCESS_KEY_ID = '7NFFU2I15W8ASBKI4J7T'
    AWS_ACCESS_SECRET_KEY = 'JGl1E8nCeItLJUvrn48y9FzdhZ+nU1+q95NsmxRH'
    AWS_REGION = 'ap-northeast-2'
else:
    OBJECT_STORAGE_ENDPOINT = 'http://172.16.11.235:9000'  # Garage 서버 주소 (기본 포트 3900)
    AWS_ACCESS_KEY_ID = 'QQY84JF5HCFNC814TE75'
    AWS_ACCESS_SECRET_KEY = 'TOzA6qIU0smDLZhQFS7N8jRUVCB+RbdYoyhtJ3Ma'
    AWS_REGION = 'ap-northeast-2'
    # Clearml 정보
    os.environ['CLEARML_WEB_HOST']='http://172.16.8.168:8080'
    os.environ['CLEARML_API_HOST']='http://172.16.8.168:8008'
    os.environ['CLEARML_FILES_HOST']='http://172.16.8.168:8081'
    os.environ['CLEARML_API_ACCESS_KEY']='NA4TVJLF4MNFPAT8JCSYBOVN0QASF2'
    os.environ['CLEARML_API_SECRET_KEY']='g3ur38Bn2GRwzTGDfcEGJy30iPv0Wx43TYDNhN4S4HjVQn5KX6OIpUbCFwTF2uOCQ3c'
    # YOLO의 자동 ClearML 로깅 비활성화
    os.environ['CLEARML_REGISTER_IGNORE'] = 'True'
    os.environ['CLEARML_SKIP_AUTO_STAGED_VCS'] = '1'
    os.environ['CLEARML_VCS_AUTO_CHECKOUT'] = '0'
    os.environ['CLEARML_VCS_IGNORE_EXTENSIONS'] = '1'
    os.environ['CLEARML_SKIP_VCS_AUTO_CONFIGURATION'] = '1'
    os.environ['CLEARML_SKIP_AUTO_STAGED_VCS'] = '1'
    os.environ['CLEARML_SKIP_VCS_AUTO_CONFIGURATION'] = '1'
    os.environ['CLEARML_SKIP_AUTO_STAGED_VCS'] = '1'
    # Git 정보를 찾지 않도록 설정
    os.environ['CLEARML_SKIP_GIT_CHECK'] = '1'
    # 원격지에서 실행 시 Git clone을 시도하지 않음
    os.environ['CLEARML_FORCE_STORE_DIFF'] = '1'
logger.info(f'breaking_point #4')
logger.info(f'phase               : {phase}')
logger.info(f'epochs_cnt          : {epochs_cnt}')
logger.info(f'work_environ        : {work_environ}')
logger.info(f'model_type_str      : {model_type_str}')
logger.info(f'process_id          : {process_id}')
logger.info(f'sub_process_id      : {sub_process_id}')
logger.info(f'project_name        : {project_name}')
logger.info(f'task_name           : {task_name}')
logger.info(f'BUCKET_NAME         : {BUCKET_NAME}')
logger.info(f'model_params        : {json.dumps(lstmae_params)}')
logger.info(f'data_params         : {json.dumps(data_params)}')
logger.info(f'db_conf             : {db_conf}')
logger.info(f'feature_cols        : {feature_cols}')
logger.info(f'data_begin_days     : {data_begin_days}')
logger.info(f'breaking_point #5')

2026-05-31 21:53:55.099 | INFO     | __main__:<module>:69 - breaking_point #1
2026-05-31 21:53:55.114 | INFO     | __main__:<module>:73 - breaking_point #2
2026-05-31 21:53:55.115 | INFO     | __main__:<module>:99 - breaking_point #3
2026-05-31 21:53:55.116 | INFO     | __main__:<module>:132 - breaking_point #4
2026-05-31 21:53:55.117 | INFO     | __main__:<module>:133 - phase               : dev
2026-05-31 21:53:55.118 | INFO     | __main__:<module>:134 - epochs_cnt          : 4
2026-05-31 21:53:55.118 | INFO     | __main__:<module>:135 - work_environ        : home
2026-05-31 21:53:55.119 | INFO     | __main__:<module>:136 - model_type_str      : LSTMAe
2026-05-31 21:53:55.120 | INFO     | __main__:<module>:137 - process_id          : P102
2026-05-31 21:53:55.121 | INFO     | __main__:<module>:138 - sub_process_id      : 1001
2026-05-31 21:53:55.122 | INFO     | __main__:<module>:139 - project_name        : mes
2026-05-31 21:53:55.123 | INFO     | __main__:<module>:140 - task_name    

db_conf = {'host': '172.21.144.1', 'port': 3306, 'user': 'mlops_detect', 'password': 'QoS908Z1!', 'database': 'mes_metric'}
table_name = P102_1001
row = <pymysql.connections.Connection object at 0x7767fada0200>
row = <pymysql.cursors.Cursor object at 0x7767fada25a0>
row = ('id', 'int', 'NO', 'PRI', None, 'auto_increment')
row = ('created_at', 'datetime', 'NO', 'MUL', None, '')
row = ('process_id', 'varchar(128)', 'YES', '', None, '')
row = ('sub_process_id', 'varchar(128)', 'YES', '', None, '')
row = ('inlet_temp_set', 'decimal(5,1)', 'YES', '', None, '')
row = ('inlet_temp_meas', 'decimal(5,1)', 'YES', '', None, '')
row = ('outlet_temp_set', 'decimal(5,1)', 'YES', '', None, '')
row = ('outlet_temp_meas', 'decimal(5,1)', 'YES', '', None, '')
row = ('chamber_dp_set', 'decimal(5,1)', 'YES', '', None, '')
row = ('chamber_dp_meas', 'decimal(5,1)', 'YES', '', None, '')
row = ('liquid_temp_meas', 'decimal(5,1)', 'YES', '', None, '')
row = ('liquid_temp_open_set', 'decimal(5,1)', 'YES', '', N

In [19]:
class IllegalFeatureException(Exception):
    """
    IllegalFeatureException
    """
    def __init__(self, msg):
        super().__init__(msg)

from torch.utils.data import Dataset

class MySQLDataSet(Dataset):
    """
    MySQLDataSet
    """
    def __init__(self, window_size: int, db_conf: dict,
                 process_id: str, sub_process_id: str,
                 base_scaler=None, limit=-1, from_days=-1,
                 mode="train"):
        self._window_size = window_size
        self._process_id = process_id
        self._sub_process_id = sub_process_id
        self._table_name = f'{process_id}_{sub_process_id}'
        self._database_name = db_conf['database'] if 'database' in db_conf else 'mes'
        self._db_conf = db_conf
        self._db_handle = self.__connect()
        self._offset_pos = 0
        self._direction = "asc"
        self._db_columns = self.__db_get_columns(table_name=self._table_name)
        self._db_columns_str = ', '.join(self._db_columns)
        # self._db_columns_str = self._db_columns_str.replace('created_at', 'UNIX_TIMESTAMP(created_at)')
        self._limit = limit
        self._current_rpt_cnt = 0
        self._from_days = from_days
        self._data_len = -1
        self._mode = mode
        self._start_offset = 0
        self._total_rows = self._get_count()
        self._scaler = self._init_scaler(base_scaler)
        logger.info(f'self._mode = {self._mode}')

    @property
    def db_columns(self) -> list:
        return self._db_columns

    @property
    def scaler(self):
        return self._scaler

    @property
    def from_days(self):
        return self._from_days

    @from_days.setter
    def from_days(self, val: int):
        self._from_days = val

    def __connect(self):
        return pymysql.connect(**self._db_conf)

    def _init_scaler(self, base_scaler):
        # 전체를 다 읽지 않고 샘플 데이터를 일부만 읽어 스케일러 기준점(Min/Max)을 잡습니다.
        try:
            with self._db_handle.cursor() as cursor:
                data_num = self._get_count()
                # logger.info(f'data_num = {data_num}')
                # limit = data_num // 10
                limit = -1
                # logger.info(f'limit = {limit}')
                query = f"SELECT {self._db_columns_str} FROM {self._table_name}"
                if self._from_days > 0:
                    query += f" where created_at > date_sub(NOW(), INTERVAL {self._from_days} DAY)"
                if limit >= 1:
                    query += f" LIMIT {limit if limit >= 1 else data_num % 10}"
                # logger.info(f'query = {query}')
                cursor.execute(query)
                rows = cursor.fetchall()
                df = pd.DataFrame(rows, columns=self._db_columns)
                if base_scaler is not None:
                    scaler = base_scaler
                    # 기존 데이터 범위를 유지하면서 새 데이터의 최솟값/최댓값을 반영하여 누적 업데이트
                    scaler.partial_fit(df.values)
                else:
                    scaler = RobustScaler()
                    scaler.fit(df.values)
                return scaler
        except Exception as ex:
            logger.error(f"Excetion : {ex}")
            raise ex

    def __db_get_columns(self, table_name: str, from_days = -1):
        query = f"""describe {table_name}"""
        columns = []
        try:
            with self._db_handle.cursor() as cursor:
                cursor.execute(query)
                rows = cursor.fetchall()
                for row in rows:
                    if row[0] not in ("id", "process_id", "sub_process_id", "is_anomaly_label", 'created_at'):
                        columns.append(row[0])
        except Exception as ex:
            logger.error(f"Exception: {ex}")
        return columns

    def _get_count(self, from_days=-1):
        query = f'select count(*) from {self._table_name}'
        if from_days > 0 or self.from_days > 0:
            query += f' where created_at > date_sub(NOW(), INTERVAL {from_days if from_days > 0 else self._from_days} DAY)'
        count = 0
        try:
            with self._db_handle.cursor() as cursor:
                cursor.execute(query)
                rows = cursor.fetchone()
                count = rows[0]
        except Exception as ex:
            logger.error(f"Exception: {ex}")
        return count

    def __len__(self):
        return self._total_rows
        # t_len = self._total_rows - self._window_size
        # return t_len if t_len >= 0 else -1

    def __get_item(self, query: str):
        try:
            with self._db_handle.cursor() as cursor:
                cursor.execute(query)
                rows = cursor.fetchall()
                return pd.DataFrame(rows, columns=self._db_columns)
        except Exception as ex:
            logger.error(f"Exception: {ex}")
        return None

    def __zero_item(self):
        in_zero_data = torch.zeros((self._window_size, len(self._db_columns)), dtype=torch.float32)
        return in_zero_data

    def __empty_item(self):
        return torch.tensor([])

    def __getitem__(self, idx):
        # logger.info(f'__get_item__ #-1 {idx}')
        if self._limit > 0 and self._current_rpt_cnt >= self._limit:
            return self.__empty_item()
        # logger.info(f'__get_item__ #1 {idx}')
        query = f'SELECT {self._db_columns_str} FROM {self._table_name}'
        if self._from_days > 0:
            query += f" where created_at > date_sub(NOW(), INTERVAL {self._from_days} DAY)"
        query += f' ORDER BY id {self._direction} LIMIT {self._window_size} OFFSET {idx}'
        sequence = self.__get_item(query=query)
        self._current_rpt_cnt += 1
        if sequence.empty or len(sequence) < self._window_size:
            return self.__zero_item()
        # 2. 정규화 및 텐서 변환
        scaled_data = self._scaler.transform(sequence.values)
        return torch.tensor(scaled_data, dtype=torch.float32)

    def close(self):
        if self._db_handle is not None:
            self._db_handle.close()
            self._db_handle = None

    def __str__(self):
        return f'MySQLDataSet(table_name=\"{self._table_name}\")'


In [20]:
run_this=True
if run_this:
    datasets = MySQLDataSet(window_size=lstmae_params['seq_len'],
                            db_conf=db_conf,
                            process_id=process_id,
                            sub_process_id=sub_process_id,
                            limit=2,
                            from_days=-1)
    logger.info(f'columns = {datasets.db_columns}')
    logger.info(f'columns = {len(datasets.db_columns)}')

    # for in_data in datasets:
    #     logger.info(f'in_data = {in_data}')
    #     if in_data is None or in_data.numel() == 0:
    #         break
    #     logger.info(f'dataset(in_data)  = {in_data}')
    datasets.close()

2026-05-31 21:53:56.179 | INFO     | __main__:__init__:38 - self._mode = train
2026-05-31 21:53:56.180 | INFO     | __main__:<module>:9 - columns = ['inlet_temp_set', 'inlet_temp_meas', 'outlet_temp_set', 'outlet_temp_meas', 'chamber_dp_set', 'chamber_dp_meas', 'liquid_temp_meas', 'liquid_temp_open_set', 'liquid_temp_close_set', 'air_broom_temp_set', 'air_broom_temp_meas', 'blower_fan_ctrl', 'exhaust_fan_ctrl', 'feed_rate_meas', 'feed_rate_ctrl', 'feed_press_meas', 'feed_press_ctrl', 'atomizer_rpm_meas', 'atomizer_rpm_ctrl', 'damper_interval_sec', 'damper_work_sec']
2026-05-31 21:53:56.181 | INFO     | __main__:<module>:10 - columns = 21


In [21]:
"""
STM-AutoEncoder 신경망 모델 정의
"""
import joblib
from sklearn.metrics import r2_score


class Encoder(nn.Module):
    def __init__(self, seq_len, n_features, inner_dim, bottleneck_dim):
        super(Encoder, self).__init__()
        self.lstm1 = nn.LSTM(n_features, inner_dim, batch_first=True, dropout=0.1)
        self.lstm2 = nn.LSTM(inner_dim, bottleneck_dim, batch_first=True)
        
    def forward(self, x):
        x, _ = self.lstm1(x)
        _, (hidden, _) = self.lstm2(x)
        return hidden.squeeze(0)


class LSTMAutoEncoder(nn.Module):
    def __init__(self,
                 process_id, sub_process_id,
                 device,
                 lstmae_params: dict=None,                 
                 train_datasets=None,
                 db_conf=None, pre_scaler_filepath_name: str = None,
                 clearml_task=None,
                 is_enable_shap=False,
                 last_train_dataset=None,
                 threshold=99,
                 verbose=1):
        super(LSTMAutoEncoder, self).__init__()
        self._process_id = process_id
        self._sub_process_id = sub_process_id
        self._lstmae_params = lstmae_params
        self._db_conf = db_conf
        self._device = device
        self._seq_len = lstmae_params['seq_len']
        self._input_size = lstmae_params['n_features']
        if verbose > 0:
            logger.info(f'lstmae_params = {lstmae_params}')
            logger.info(f'db_conf       = {db_conf}')
            logger.info(f'device        = {device}')
        self._encoder = nn.ModuleDict({
            'lstm1': nn.LSTM(self._input_size, lstmae_params['inner_dim'], batch_first=True),
            'lstm2': nn.LSTM(lstmae_params['inner_dim'], self._input_size, batch_first=True)
        })
        # 2. Decoder 구조 (동일하게 lstm1, lstm2 구조)
        self._decoder = nn.ModuleDict({
            'lstm1': nn.LSTM(self._input_size,  lstmae_params['inner_dim'], batch_first=True),
            'lstm2': nn.LSTM(lstmae_params['inner_dim'], self._input_size, batch_first=True)
        })
        self._fc = nn.Linear(lstmae_params['inner_dim'], self._input_size)
        self._train_datasets = train_datasets
        self._clearml_task = clearml_task
        self._base_scaler = None
        self._threshold = threshold
        if pre_scaler_filepath_name is not None:
            self._base_scaler = joblib.load(pre_scaler_filepath_name)
        self._threshold = 99
        if is_enable_shap and last_train_dataset is not None:
            num_bg_samples = 10
            train_datasets = self.__string_to_tensor(last_train_dataset)
            train_datasets = train_datasets.to(torch.float32)
            logger.info(f"최종 변환된 타입: {train_datasets.dtype}") # torch.float32
            X_train_flattened = train_datasets[:num_bg_samples].reshape(num_bg_samples, -1)
            X_train_flattened = np.asarray(X_train_flattened, dtype=np.float32)
            if isinstance(X_train_flattened, torch.Tensor):
                X_train_flattened = X_train_flattened.detach().cpu().numpy()            
            if X_train_flattened is not None:
                logger.info(f'X_train_flattened={type(X_train_flattened)}, {len(X_train_flattened)}, {X_train_flattened}')
                self.__explainer = shap.KernelExplainer(self.__model_moment_error_wrapper, X_train_flattened)
            else:
                self.__explainer = None
        else:
            self.__explainer = None

    @property
    def threshold(self):
        return self._threshold

    @property
    def train_datasets(self):
        return self._train_datasets

    def forward(self, x):
        # ------------------------------------------------------------------
        # 1. Encoder 단계
        # ------------------------------------------------------------------
        # lstm1 통과 -> out1 차원: [Batch, Timesteps, inner_dim]
        timesteps = self._seq_len
        out1, _ = self._encoder['lstm1'](x)
        
        # lstm2 통과 -> out2 차원: [Batch, Timesteps, input_size]
        # h_n(마지막 은닉 상태) 추출 -> 차원: [1, Batch, input_size]
        out2, (h_n, _) = self._encoder['lstm2'](out1)
        
        # ------------------------------------------------------------------
        # 2. Bottleneck (Repeat Vector) 단계
        # Decoder의 첫 번째 입력으로 넣기 위해, Encoder의 마지막 출력을 복사 확장합니다.
        # ------------------------------------------------------------------
        # h_n 차원 변경: [1, Batch, input_size] -> [Batch, Timesteps, input_size]
        # self.timesteps는 윈도우 크기(60)입니다. 클래스 변수로 정의되어 있어야 합니다.
        decoder_input = h_n.repeat(timesteps, 1, 1).transpose(0, 1)

        # ------------------------------------------------------------------
        # 3. Decoder 단계
        # ------------------------------------------------------------------
        # lstm1 통과 -> dec_out1 차원: [Batch, Timesteps, inner_dim]
        dec_out1, _ = self._decoder['lstm1'](decoder_input)
        
        # lstm2 통과 -> dec_out2 차원: [Batch, Timesteps, input_size]
        dec_out2, _ = self._decoder['lstm2'](dec_out1)

        # ------------------------------------------------------------------
        # 4. Linear (Dense) 단계
        # _fc(Linear) 계층이 'inner_dim'을 입력받도록 설계되어 있으므로,
        # dec_out1(inner_dim 차원)을 넣어서 최종 복원값을 계산합니다.
        # ------------------------------------------------------------------
        # final_output 차원: [Batch, Timesteps, input_size]
        final_output = self._fc(dec_out1)
        
        return final_output

    def decision_function(self, x_tensor):
        self.eval() # 추론 모드 고정
        with torch.no_grad():
            # 1. 모델 예측 (재구성)
            predictions = self.forward(x_tensor)
            # 2. 차원 1(시퀀스), 2(특성)에 대해 평균 MSE 오차 계산
            # mse shape: [Batch_size]
            # logger.info(f'decision_function x_tensor={x_tensor}')
            # logger.info(f'decision_function predictions={predictions}')
            mse = torch.mean((x_tensor - predictions) ** 2, dim=(1, 2))
            # logger.info(f'decision_function mse={mse}')
        return mse

    def __do_batch(self, batch, optimizer, criterion):
        inputs = batch[0].to(self._device)
        optimizer.zero_grad()
        outputs = self.forward(x=inputs)
        loss = criterion(outputs, inputs)
        loss.backward()
        optimizer.step()
        loss = loss.item() * inputs.size(0)
        inputs = inputs.detach().cpu().numpy()
        outputs = outputs.detach().cpu().numpy()
        return inputs, outputs, loss

    def set_threshold(self, optimizer, criterion):
        self.eval()
        all_errors = []
        with torch.no_grad():
            # 메모리 안전을 위해 배치 단위로 오차(MSE) 수집
            eval_loader = DataLoader(self._train_datasets, batch_size=self._lstmae_params['batch_size'], shuffle=False)
            epoch_loss = 0.0            
            for batch in eval_loader:
                batch_x = batch[0].to(self._device)
                outputs = self.forward(x=batch_x)
                # 각 샘플별 MSE 계산 (차원 1, 2 평균)
                mse = torch.mean((batch_x - outputs) ** 2, dim=(1, 2))
                all_errors.extend(mse.cpu().numpy())
        # 수집된 오차 배열에서 상위 95% 지점 산출
        calculated_threshold = np.percentile(all_errors, 95)
        self._threshold = float(calculated_threshold)
        logger.info(f"🎯 95% 백분위수 임계치 결정 및 저장 완료: {self._threshold:.6f}")

    def set_threshold_by_static(self, train_x_tensor):
        """
        [임계치 설정용 메서드]
        정상 데이터 텐서를 받아 통계적 기법(평균 + 3*표준편차)으로 임계치 텐서를 자동 계산합니다.
        """
        train_errors = self.decision_function(train_x_tensor)
        
        # 순수 텐서 연산으로 평균과 표준편차 계산 후 임계치 지정
        mean_val = torch.mean(train_errors)
        std_val = torch.std(train_errors)
        
        self._threshold = mean_val + (3 * std_val)

    def execute_train(self):
        """
        ClearML 자산 로드 및 점진적 재학습 실행
        """
        logger.info(f"사용중인 연산 디바이스: {self._device}")
        result = False
        last_dataset = None

        self._train_loader = DataLoader(self._train_datasets, batch_size=self._lstmae_params['batch_size'], shuffle=True, drop_last=True)
        # 3) 모델 트레이닝 루프
        current_task = Task.current_task()
        clearml_logger = current_task.get_logger() if current_task else None
        self.to(self._device)
        logger.info("훈련 시작 ___________________________ #1")
        self.train()
        criterion = nn.MSELoss()
        optimizer = optim.Adam(self.parameters(), lr=self._lstmae_params['learning_rate'])
        try:
            all_losses = []
            for epoch in range(1, (self._lstmae_params['epochs']+1)):
                epoch_loss = 0.0
                logger.info(f'epoch = {epoch} / {self._lstmae_params['epochs']}')
                # 에폭마다 정확도 계산을 위해 원본(정답)과 복원본을 담을 임시 리스트
                batch_cnt = 0
                logger.info(f'###step #-3')
                for batch in self._train_loader:
                    logger.info(f'###step #-2')
                    logger.info(f'###step #-2 batch={batch[0][0]}')
                    inputs = batch[0].to(self._device)
                    optimizer.zero_grad()
                    outputs = self.forward(x=inputs)
                    loss = criterion(outputs, inputs)
                    loss.backward()
                    optimizer.step()
                    logger.info(f'###step #-2.1')                    
                    epoch_loss +=(loss.item() * inputs.size(0))
                    logger.info(f'###step #-1')
                    batch_cnt += 1
                avg_loss = epoch_loss / len(self._train_datasets)
                all_losses.append(avg_loss)
                logger.info(f'###step $1')
                if clearml_logger:
                    clearml_logger.report_scalar(
                        title="Loss",            # 대시보드 탭의 대분류 이름
                        series="MSE",            # 꺾은선 그래프 범례 이름
                        value=float(avg_loss), # Y축 값
                        iteration=epoch          # X축 값 (에폭 번호)
                    )
            # 실무 표준 기법: 정상 데이터 오차의 상위 5% 지점(95 분위수)을 임계치로 설정
            # (만약 완벽히 깨끗한 정상 데이터만 있다면 99 분위수나 최대값을 사용해도 좋습니다)
            logger.info("최적 임계치(Threshold) 계산__________")
            last_dataset = self.__get_last_train_data(dataloader=self._train_loader)
            self.set_threshold(optimizer=optimizer, criterion=criterion)

            # ClearML 하이퍼파라미터(Configuration) 영역에 임계값과 마지막 훈련 데이터셋 업데이트
            if self._clearml_task:
                last_dataset = self.__tensor_to_string(last_dataset)
                self._clearml_task.connect_configuration({"computed_threshold": float(self._threshold)}, name="Model_Threshold")
                self._clearml_task.connect_configuration({"last_datasets": last_dataset}, name="Last_Train_Dataset")
            result = True
        except Exception as ex:
            logger.error(f'Exception : {ex}')
        logger.info(f"--- 훈련 완료 --- [result={result}]")
        return result, last_dataset

    def __get_last_train_data(self, dataloader):
        last_batch = None
        for batch in dataloader:
            last_batch = batch  # 루프가 끝나면 자동으로 맨 마지막 배치가 남습니다.
        # 2. 마지막 배치에서 X(입력 데이터)와 y(타깃) 분리
        # (Dataset 구조에 따라 batch가 X 단독이거나 (X, y) 튜플일 수 있습니다)
        if isinstance(last_batch, (list, tuple)):
            last_X, last_y = last_batch[0], last_batch[1]
        else:
            last_X = last_batch
        return last_X

    def __calculate_anomaly_threshold_torch(self, method='percentile', q=99, std_dev=3):
        # 1. 디바이스 맞추기 (GPU에 있다면 GPU에서 연산 수행)
        y_true = None
        for batch in self._train_loader:
            y_true = batch.to(self._device)
        logger.info(y_true[0])
        device = y_true.device
        y_pred = self(y_true)
        y_pred = y_pred.to(device)
    
        # 2. 각 샘플별로 재구성 오차(MAE) 계산
        # 0번 축(Samples)을 제외한 모든 축의 평균을 구함
        dimensions = list(range(1, y_true.dim()))
        mae_loss = torch.mean(torch.abs(y_true - y_pred), dim=dimensions)
    
        # 3. 지정된 방식으로 임계치 계산
        if method == 'percentile':
            # torch.quantile은 0~1 사이의 비율을 받으므로 q를 100으로 나눕니다.
            q_fraction = q / 100.0
            threshold = torch.quantile(mae_loss, q_fraction)
        elif method == 'sigma':
            # 통계적 접근: 평균 + (std_dev * 표준편차)
            threshold = torch.mean(mae_loss) + (std_dev * torch.std(mae_loss))
        else:
            raise ValueError("method는 'percentile' 또는 'sigma' 중 하나여야 합니다.")
        return threshold, mae_loss

    def __tensor_to_string(self, tensor_data):
        import torch
        import io
        import base64

        # Memory Buffer에 텐서 직렬화
        buffer = io.BytesIO()
        torch.save(tensor_data, buffer)
        # 바이트 데이터를 문자열(String)로 인코딩
        tensor_string = base64.b64encode(buffer.getvalue()).decode('utf-8')
        return tensor_string

    def __string_to_tensor(self, string_data):
        import base64
        import io
        tensor_bytes = base64.b64decode(string_data.encode('utf-8'))
        buffer = io.BytesIO(tensor_bytes)
        # 원래 텐서로 복원
        restored_tensor = torch.load(buffer, weights_only=True)
        return restored_tensor
    
    def save_output(self, out_path: str, version: str=None):
        logger.info(f'out_path={out_path}')
        if os.path.isdir(out_path) == False:
            os.mkdir(out_path)
        weight_filepath_name = f'{out_path}/lstm_ae_weights{"" if version is None else "_" + version}.pth'
        scaler_filepath_name = f'{out_path}/minmax_scaler{"" if version is None else "_" + version}.pkl'        
        torch.save(self.state_dict(), weight_filepath_name)
        joblib.dump(self._train_datasets.scaler, scaler_filepath_name)
        logger.info("가중치('lstm_ae_weights.pth') 및 스케일러('minmax_scaler.pkl') 파일 추출 완료!")
        weight_filepath_name, scaler_filepath_name
        return weight_filepath_name, scaler_filepath_name


In [22]:
# 이전 모델 정보 다운 로드

def download_artifacts_from_clearml(task_id):
    """
    ClearML 서버에서 특정 프로젝트 및 태스크 이름에 맵핑된 
    가중치와 스케일러 파일을 찾아 로컬에 다운로드합니다.
    """
    task = Task.get_task(task_id=task_id)
    
    if task is None:
        raise ValueError(f"지정한 프로젝트 또는 태스크를 ClearML에서 찾을 수 없습니다.")

    # 2) Task에 등록된 artifacts 딕셔너리에서 파일들을 가져옵니다.
    artifacts = task.artifacts
    
    if "lstm_autoencoder_weights" not in artifacts or "minmax_scaler" not in artifacts:
        logger.info("태스크 내에 'lstm_autoencoder_weights' 또는 'minmax_scaler' Artifact가 존재하지 않습니다.")
        return None, None

    # 3) get_local_copy() 함수를 호출하면 ClearML 원격 스토리지(S3 등)에서 
    #    현재 로컬 머신의 임시 캐시 디렉토리로 파일을 자동 다운로드하고 그 절대경로를 반환합니다.
    local_weight_path = artifacts["lstm_autoencoder_weights"].get_local_copy()
    local_scaler_path = artifacts["minmax_scaler"].get_local_copy()
    
    logger.info("ClearML로부터 다운로드 성공!")
    logger.info(f"-> 가중치 로컬 경로: {local_weight_path}")
    logger.info(f"-> 스케일러 로컬 경로: {local_scaler_path}")
    
    return local_weight_path, local_scaler_path


def get_pretrained_values_from_db(project_name, process_id, sub_process_id, db_config):
    import pymysql
    with pymysql.connect(host=db_config['host'], port=db_config['port'],
                         user=db_config['user'], password=db_config['password'],
                         database="mlops_detect") as conn:
        with conn.cursor() as cursor:
            select_query = f"""
                            select weight, scaler from model_info where
                                project_name="{project_name}" and
                                process_id="{process_id}" and
                                sub_process_id="{sub_process_id}" and
                                weight is not null and
                                scaler is not null
                            """
            cursor.execute(query=select_query)
            row = cursor.fetchone()
            if row is not None:
                file_name_prefix = f"{project_name}_{process_id}_{sub_process_id}"
                weight_file = f"{file_name_prefix}.pth"
                scaler_file = f"{file_name_prefix}.scaler"
                with open(weight_file, "wb") as fd:
                    fd.write(row[0])
                with open(scaler_file, "wb") as fd:
                    fd.write(row[1])
                return weight_file, scaler_file
    return None, None


def get_pretrained_weigth(project_name, task_name):
    """
    마지막으로 훈련에 성공한 모델의 가중치 파일 다운 받습니다.
    """
    tasks = Task.query_tasks(
        project_name=project_name,
        task_name=task_name,
        task_filter={
            'status': ['completed'],
            'order_by': ['-created']  # Ensures the most recently updated is first
        }
    )
    logger.info(f'tasks\'len = {len(tasks)}')
    if len(tasks) > 0:
        last_task_id= tasks[0]
        logger.info(f'last_task_id = {last_task_id}')
        local_weight_path, local_scaler_path = download_artifacts_from_clearml(task_id=last_task_id)
    logger.info(f"Weight downloaded to #3: {local_weight_path}")
    logger.info(f"Scaler downloaded to #3: {local_scaler_path}")
    return local_weight_path, local_scaler_path


In [23]:
def train_and_extract_weights(lstmae_params: dict,
                              pre_weight_filepath_name: str=None,
                              pre_scaler_filepath_name: str=None,
                              clearml_task=None,
                              version: str=None):
    """
    ClearML 자산 로드 및 점진적 재학습 실행    
    """
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    logger.info(f"사용중인 연산 디바이스: {device}")

    try:
        base_scaler = None
        if pre_scaler_filepath_name is not None:
            from pathlib import Path
            file_path = Path(pre_scaler_filepath_name)
            if file_path.is_file():
                base_scaler = joblib.load(pre_scaler_filepath_name)                
            else:
                logger.warning(f'파일이 존재하지 않습니다. {pre_scaler_filepath_name}')
        # 1) 데이터셋 및 데이터로더 초기화
        train_datasets = MySQLDataSet(window_size=lstmae_params['seq_len'],
                                      db_conf=db_conf,
                                      process_id=process_id,
                                      sub_process_id=sub_process_id,
                                      base_scaler=base_scaler,
                                      mode="train",
                                      limit=-1)
        model = LSTMAutoEncoder( process_id=process_id, sub_process_id=sub_process_id,
                                 lstmae_params=lstmae_params, train_datasets=train_datasets,
                                 device=device,
                                 db_conf=db_conf, pre_scaler_filepath_name=pre_scaler_filepath_name,
                                 clearml_task=Task.current_task() if clearml_task is None else clearml_task).to(device)
        if pre_weight_filepath_name is not None:
            logger.info(f'pre_weight_filepath_name = {pre_weight_filepath_name}')
            from pathlib import Path
            file_path = Path(pre_weight_filepath_name)
            if file_path.is_file():
                model.load_state_dict(torch.load(pre_weight_filepath_name, map_location=device))
            else:
                logger.warning(f'파일이 존재하지 않습니다. {pre_weight_filepath_name}')
        result, last_dataset = model.execute_train()
        train_datasets.close()
        logger.info(f'train result = {result}')
        logger.info(f'train threshold = {model.threshold}')
        if result:
            weight_filepath_name, scaler_filepath_name = model.save_output(out_path='weights', version=version)
        else:
            weight_filepath_name = scaler_filepath_name = None
        return weight_filepath_name, scaler_filepath_name, model.threshold, last_dataset
    except Exception as ex:
        logger.error(f'Exception: {ex}')
        if train_datasets is not None:
            train_datasets.close()

In [24]:
logger.info(f'project_name = {project_name}')
logger.info(f'task_name    = {task_name}')
task = None
run_this = True
if run_this == False:
    exit()
logger.info(f'tracking #-1')

is_remote_agent = os.environ.get("CLEARML_PROC_MASTER_ID") is not None or running_remotely()

if use_clearml:
    if is_remote_agent:
        # 1. 원격 도커 내부에서는 절대로 새로 init하지 않고, 전달받은 기존 태스크를 '재사용'합니다.        
        task = Task.init(project_name=project_name,
                         task_name=task_name,
                         reuse_last_task_id=True,
                         continue_last_task=True,
                         task_type=Task.TaskTypes.training)
    else:
        # 2. 로컬(최초 실행) 환경일 때만 완전히 새로운 태스크를 만듭니다.
        task = Task.init(project_name=project_name,
                         task_name=task_name,
                         reuse_last_task_id=False,    # 로컬에서는 새로 생성
                         continue_last_task=False,
                         task_type=Task.TaskTypes.training)
    if len(lstmae_params.keys()) > 0:
        task.connect(lstmae_params)
    task.set_task_type('training')
    task.set_parameter('version', task_version)
    task.add_tags(task_version)
    task.connect_configuration({"model_version": task_version}, name="Version_Control")    
    ia_task_initiated = False
    task.set_base_docker(
        docker_image=docker_image,  # 원격에서 실행할 베이스 이미지
        docker_arguments="--privileged --device /dev/fuse"  # 도커 실행 인자
    )

    if phase == "prod":
        task.execute_remotely(queue_name='services',
                              clone=False)
        if not running_remotely():
            # [로컬 Parent 프로세스 영역]
            logger.info("Parent: 자식 태스크를 원격 에이전트에 전달했습니다.")
            task.close()
            task.delete(delete_artifacts_and_models=True)
            exit()
if use_clearml:
    pre_weight_filepath_name, pre_scaler_filepath_name = get_pretrained_weigth(project_name=project_name,
                                                                               task_name=task_name)
else:
    pre_weight_filepath_name, pre_scaler_filepath_name = get_pretrained_values_from_db(project_name=project_name, 
                                                                                       process_id=process_id,
                                                                                       sub_process_id=sub_process_id,
                                                                                       db_config=db_conf)
logger.info(f'pre_weight_filepath_name = {pre_weight_filepath_name}')
logger.info(f'pre_scaler_filepath_name = {pre_scaler_filepath_name}')
weight_filepath_name, scaler_filepath_name, threshold, train_datasets = train_and_extract_weights(lstmae_params=lstmae_params,
                                                                                                  pre_weight_filepath_name=pre_weight_filepath_name,
                                                                                                  pre_scaler_filepath_name=pre_scaler_filepath_name,
                                                                                                  clearml_task=task,
                                                                                                  version=task_version)

2026-05-31 21:53:58.838 | INFO     | __main__:<module>:1 - project_name = mes
2026-05-31 21:53:58.839 | INFO     | __main__:<module>:2 - task_name    = P102_1001
2026-05-31 21:53:58.839 | INFO     | __main__:<module>:7 - tracking #-1
2026-05-31 21:53:58.851 | INFO     | __main__:<module>:55 - pre_weight_filepath_name = mes_P102_1001.pth
2026-05-31 21:53:58.852 | INFO     | __main__:<module>:56 - pre_scaler_filepath_name = mes_P102_1001.scaler
2026-05-31 21:53:58.856 | INFO     | __main__:train_and_extract_weights:10 - 사용중인 연산 디바이스: cpu
2026-05-31 21:53:58.966 | INFO     | __main__:__init__:38 - self._mode = train
2026-05-31 21:53:58.967 | INFO     | __main__:__init__:41 - lstmae_params = {'seq_len': 20, 'n_features': 21, 'inner_dim': 64, 'bottleneck_dim': 16, 'batch_size': 64, 'learning_rate': 0.001, 'dropout': 0.1, 'epochs': 4}
2026-05-31 21:53:58.967 | INFO     | __main__:__init__:42 - db_conf       = {'host': '172.21.144.1', 'port': 3306, 'user': 'mlops_detect', 'password': 'QoS908Z

In [25]:
def write_pretrained_values_to_db(project_name, process_id,
                                  sub_process_id, db_config,
                                  weight, scaler,
                                  threshold, train_datasets,
                                  version):
    import pymysql
    with pymysql.connect(host=db_config['host'], port=db_config['port'],
                         user=db_config['user'], password=db_config['password'],
                         database="mlops_detect") as conn:
        train_datasets = str(train_datasets.tolist())
        with conn.cursor() as cursor:
            select_query = f"""
                            select count(*) from model_info where
                                project_name="{project_name}" and
                                process_id="{process_id}" and
                                sub_process_id="{sub_process_id}" and
                                weight is not null and
                                scaler is not null and
                                version="{version}"
                            """
            # logger.info(f'query = {select_query}')
            # logger.info(f'train_datasets = {len(train_datasets)}{train_datasets}')            
            cursor.execute(query=select_query)
            row = cursor.fetchone()
            if row[0] == 0:
                columns = ["project_name", "process_id", "sub_process_id", "weight",
                           "scaler", "threshold", "train_datasets", "version"]
                insert_query = f"""
                                insert into model_info({", ".join(columns)})
                                    values(%s, %s, %s, %s, %s, %s, %s, %s)
                                """
                data = (project_name, process_id, sub_process_id, weight,
                        scaler, str(threshold), train_datasets, version)
                logger.info(f'query = {insert_query}')
                cursor.execute(insert_query, data)
            else:
                update_query = f"""
                                update model_info set weight=%s, scaler=%s,
                                                      threshold=%s, train_datasets=%s
                                       where project_name=%s and sub_process_id=%s and
                                          process_id=%s and version=%s
                                """
                data = (weight, scaler, threshold,
                        train_datasets, project_name, 
                        sub_process_id, process_id,
                        version)
                logger.info(f'query          = {update_query}')
                cursor.execute(update_query, data)
            conn.commit()

## 가중치 파일과 스케일 파일 저장 및 종료
delete_file = False
if use_clearml:
    try:
        if weight_filepath_name:
            task.upload_artifact(name="lstm_autoencoder_weights", artifact_object=weight_filepath_name)
            logger.info(f"모델이 파일로 저장되었습니다. [{weight_filepath_name}]")
        if scaler_filepath_name:
            task.upload_artifact(name="minmax_scaler", artifact_object=scaler_filepath_name)
        time.sleep(10)
        if delete_file:
            if weight_filepath_name:
                os.remove(weight_filepath_name)
            if scaler_filepath_name:
                os.remove(scaler_filepath_name)
    except Exception as ex:
        logger.info(f'Exception : {ex}')
    finally:
        task.close()
else:
    # weight_filepath_name, scaler_filepath_name, threshold, train_datasets
    weight = None
    scaler = None
    with open(weight_filepath_name, "rb") as fd:
        weight = fd.read()

    with open(scaler_filepath_name, "rb") as fd:
        scaler = fd.read()
    logger.info(f'threshold = {threshold}')
    write_pretrained_values_to_db(project_name=project_name, process_id=process_id,
                                  sub_process_id=sub_process_id, db_config=db_conf,
                                  weight=weight, scaler=scaler,
                                  threshold=float(threshold), train_datasets=train_datasets,
                                  version=task_version)

2026-05-31 21:58:09.731 | INFO     | __main__:<module>:79 - threshold = 0.02522958070039749
2026-05-31 21:58:09.746 | INFO     | __main__:write_pretrained_values_to_db:47 - query          = 
                                update model_info set weight=%s, scaler=%s,
                                                      threshold=%s, train_datasets=%s
                                       where project_name=%s and sub_process_id=%s and
                                          process_id=%s and version=%s
                                


## 주의

여기서 부터는 clearml의 execute-remotely를 위한 코드 입니다.

In [ ]:
# THIS_SKIP_START

In [ ]:
def convert_ipynb_to_py(file_name):
    import json
    from loguru import logger
    from pathlib import Path
    import os

    skip_line = False
    this_skip_start = "# THIS_SKIP_START"
    this_skip_end = "# THIS_SKIP_END"

    root_dir = str(Path.home())
    python_file_name = file_name.replace('ipynb', 'py')
    py_filepath_name = f"{root_dir}{os.sep}{python_file_name}"
    
    logger.info(f'ipynb = {file_name}')
    logger.info(f'py    = {python_file_name}')
    with open(file_name, "r", encoding="utf-8") as f:
        nb_data = json.load(f)
    py_lines = []
    for cell in nb_data.get("cells", []):
        if cell.get("cell_type") == "code":
            for line in cell.get("source", []):
                if not line.strip().startswith(("%", "!", )):
                    #lines = [l for l in cell.get("source", []) if not l.strip().startswith(("%", "!", ))]
                    if line == this_skip_start:
                        skip_line = True
                    elif line == this_skip_end:
                        skip_line = False
                    if skip_line == False:
                        py_lines.append(line)
            py_lines.append("\n\n")
    logger.info(f'py_filepath_name = {py_filepath_name}')
    with open(py_filepath_name, "wt") as fd:
        for line in py_lines:
            fd.write(line)
    return py_filepath_name

In [ ]:
def run_python_code(py_file_name, project_name, process_id, sub_process_id):
    import os
    import subprocess
    import sys
    import json
    from loguru import logger
    # rclone 설정 (환경 변수 방식)
    run_env = os.environ.copy()
    run_env["PHASE"] = os.environ.get("PHASE", "prod")
    run_env["EPOCHS_CNT"] = str(os.environ.get("EPOCHS_CNT", 1))
    run_env["DOCKER_IMAGE"] = os.environ.get("DOCKER_IMAGE", "172.16.11.236:5000/spire/python:3.12-bullseye")
    run_env["TASK_VERSION"] = os.environ.get("TASK_VERSION", "1.0")
    run_env["DATA_BEGIN_DAYS"] = str(os.environ.get("DATA_BEGIN_DAYS", -1))
    run_env["PROJECT_NAME"] = os.environ.get("PROJECT_NAME", project_name)
    run_env["MODEL_TYPE_STR"] = os.environ.get("MODEL_TYPE_STR", "LSTMAe")
    run_env["PROCESS_ID"] = os.environ.get("PROCESS_ID", process_id)
    run_env["SUB_PROCESS_ID"] = os.environ.get("SUB_PROCESS_ID", sub_process_id)
    run_env["OUT_FEATURE_COLS"] = os.environ.get("OUT_FEATURE_COLS", "inlet_temp_meas")
    run_env["DB_CONF"] = os.environ.get("DB_CONF", '{"host":"172.16.9.60", "port": 3306, "user": "mlops_detect", "password": "QoS908Z1!", "database": "mes_metric"}')
    lstmae_def_params = {
        'seq_len': 20,            # 시퀀스 길이 (예: 1분 단위 수집 시 1시간 분량)
        'n_features': n_features, # 센서 개수 (예: 온도, 습도, 진동)
        'inner_dim': 64,          # LSTM 은닉층 차원
        'bottleneck_dim': 16,     # 압축 차원 (특징 추출 공간)
        'batch_size': 64,         # 배치 크기
        'learning_rate': 0.001,   # 초기 학습률
        'dropout': 0.1,           # 시계열 정보 유지를 위한 낮은 드롭아웃
        'epochs': 20              # 학습 에폭 수
    }
    run_env["MODEL_PARAMS"] = os.environ.get("MODEL_PARAMS", json.dumps(lstmae_def_params))
    data_def_params = {
        'split_ratio': 0.9
    }
    run_env["DATA_PARAMS"] = os.environ.get("DATA_PARAMS", json.dumps(data_def_params))
    
    command = [sys.executable,
               py_file_name]
    logger.info(f'command = {command}')
    process = subprocess.run(command, text=True, env=run_env)
    # process = subprocess.run(command, text=True, env=run_env, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

In [14]:
def remove_file(py_filepath_name):
    import os
    import time
    # 파일이 존재하는지 확인 후 안전하게 삭제
    time.sleep(10)
    if os.path.exists(py_filepath_name):
        os.remove(py_filepath_name)
        print("파일 삭제 완료")
    else:
        print("파일이 존재하지 않습니다.")

In [ ]:
NOTEBOOK_NAME = "P102_1001_LSTMAe.ipynb"
task_name = "P102_1001"
py_filepath_name = convert_ipynb_to_py(file_name=NOTEBOOK_NAME)
run_python_code(py_file_name=py_filepath_name, project_name="mes", process_id="P102", sub_process_id="1001")
import os
os._exit(0)
# remove_file(py_filepath_name=py_filepath_name)

2026-05-29 15:05:42.595 | INFO     | __main__:convert_ipynb_to_py:15 - ipynb = P102_1001_LSTMAe.ipynb
2026-05-29 15:05:42.597 | INFO     | __main__:convert_ipynb_to_py:16 - py    = P102_1001_LSTMAe.py
2026-05-29 15:05:42.601 | INFO     | __main__:convert_ipynb_to_py:32 - py_filepath_name = /home/jupyter/P102_1001_LSTMAe.py
2026-05-29 15:05:42.603 | INFO     | __main__:run_python_code:38 - command = ['/opt/jupyter/kernel/clearml/bin/python', '/home/jupyter/P102_1001_LSTMAe.py']
2026-05-29 15:05:45.800 | INFO     | __main__:<module>:69 - breaking_point #1
2026-05-29 15:05:45.819 | INFO     | __main__:<module>:73 - breaking_point #2
2026-05-29 15:05:45.819 | INFO     | __main__:<module>:99 - breaking_point #3
2026-05-29 15:05:45.819 | INFO     | __main__:<module>:132 - breaking_point #4
2026-05-29 15:05:45.819 | INFO     | __main__:<module>:133 - phase               : prod
2026-05-29 15:05:45.819 | INFO     | __main__:<module>:134 - epochs_cnt          : 1
2026-05-29 15:05:45.820 | INFO  

db_conf = {'host': '172.16.9.60', 'port': 3306, 'user': 'mlops_detect', 'password': 'QoS908Z1!', 'database': 'mes_metric'}
table_name = P102_1001
row = <pymysql.connections.Connection object at 0x7f64feff06e0>
row = <pymysql.cursors.Cursor object at 0x7f64ffbe7140>
row = ('id', 'int', 'NO', 'PRI', None, 'auto_increment')
row = ('created_at', 'datetime', 'NO', 'MUL', None, '')
row = ('process_id', 'varchar(128)', 'YES', '', None, '')
row = ('sub_process_id', 'varchar(128)', 'YES', '', None, '')
row = ('inlet_temp_set', 'decimal(5,1)', 'YES', '', None, '')
row = ('inlet_temp_meas', 'decimal(5,1)', 'YES', '', None, '')
row = ('outlet_temp_set', 'decimal(5,1)', 'YES', '', None, '')
row = ('outlet_temp_meas', 'decimal(5,1)', 'YES', '', None, '')
row = ('chamber_dp_set', 'decimal(5,1)', 'YES', '', None, '')
row = ('chamber_dp_meas', 'decimal(5,1)', 'YES', '', None, '')
row = ('liquid_temp_meas', 'decimal(5,1)', 'YES', '', None, '')
row = ('liquid_temp_open_set', 'decimal(5,1)', 'YES', '', No

2026-05-29 15:05:47.537 | INFO     | __main__:<module>:790 - run in phase = prod


In [ ]:
# THIS_SKIP_END